# Laboratorio 3 · Classificare i guasti

**ITS Academy Cremona** · Corso Intelligenza Artificiale e Machine Learning · Digitalizzazione dei processi industriali

Lezione 3 (ore 5–6) · Classificare e raggruppare · Docente: Rosario Marra

**Durata:** 45 minuti · **Modalità:** A coppie

**Nome e cognome:** _scrivi qui_  
**Compagno/a di lavoro (se in coppia):** _scrivi qui_

## Obiettivo

Addestrare classificatori su dati sbilanciati, leggere la matrice di confusione, scegliere la soglia in base ai costi e confrontare albero e foresta.

## Prima di iniziare

1. **File › Salva una copia su Drive**: lavora sulla tua copia.
2. Esegui le celle **in ordine**, dall’alto in basso (Maiusc + Invio).
3. Leggi ogni output: le domande finali si basano su quello che vedi.

## I passi

1. **Prepara i dati.** Esegui caricamento, divisione e normalizzazione.
2. **Il modello «pigro».** Calcola l’accuratezza di un modello che risponde sempre «nessun guasto». Sorpreso?
3. **Regressione logistica.** Addestrala con le classi bilanciate e leggi matrice di confusione, precisione e richiamo.
4. **La soglia giusta.** Un guasto non visto costa 5.000 €, un fermo inutile 300 €: confronta le soglie 0,3 · 0,5 · 0,7.
5. **Albero e foresta.** Disegna un albero di profondità 3, traduci una regola a parole e confronta il richiamo con una random forest.
6. **Facoltativo: k-means.** Trova 3 gruppi di funzionamento e controlla in quale si concentrano i guasti.

## 1. Caricamento

In [ ]:
!pip install ucimlrepo -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt

# AI4I 2020 Predictive Maintenance (UCI, licenza CC BY 4.0): 10.000 misure di una macchina.
# Se il sito UCI non risponde, i dati arrivano dalla copia di riserva sul sito del corso.
COPIA = "https://rosariomarra.github.io/corsi/ia-ml/materiali/ai4i2020.csv"
try:
    from ucimlrepo import fetch_ucirepo
    df = fetch_ucirepo(id=601).data.original.copy()
    print("Dati scaricati dall'UCI")
except Exception:
    df = pd.read_csv(COPIA, encoding="utf-8-sig")
    print("UCI non raggiungibile: uso la copia di riserva")
df.columns = [c.split(" [")[0] for c in df.columns]   # toglie le unità dai nomi delle colonne
df.head()

## 2. Training, test e normalizzazione

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

feature = ["Air temperature", "Process temperature", "Rotational speed", "Torque", "Tool wear"]
X, y = df[feature], df["Machine failure"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = MinMaxScaler()
X_train_n = scaler.fit_transform(X_train)   # impara minimo e massimo SOLO sul training
X_test_n = scaler.transform(X_test)         # applica gli stessi valori al test
print(X_train.shape, X_test.shape)

## 3. Modello pigro e regressione logistica

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("Modello pigro:", accuracy_score(y_test, [0] * len(y_test)))

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train_n, y_train)
pred = clf.predict(X_test_n)
print(confusion_matrix(y_test, pred))      # [[VN FP] [FN VP]]
print(classification_report(y_test, pred, digits=3))

## 4. Soglia e costi

In [ ]:
prob = clf.predict_proba(X_test_n)[:, 1]
for soglia in [0.3, 0.5, 0.7]:
    p = (prob >= soglia).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_test, p).ravel()
    costo = fn * 5000 + fp * 300
    print(f"soglia {soglia}:  VP={vp}  FN={fn}  FP={fp}   costo stimato {costo:,} €")

## 5. Albero e foresta

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score

albero = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=0).fit(X_train, y_train)
plt.figure(figsize=(18, 7))
plot_tree(albero, feature_names=feature, class_names=["ok", "guasto"], filled=True); plt.show()

foresta = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0).fit(X_train, y_train)
for nome, m in [("albero", albero), ("foresta", foresta)]:
    print(nome, "- richiamo guasti:", round(recall_score(y_test, m.predict(X_test)), 3))
print(pd.Series(foresta.feature_importances_, index=feature).sort_values(ascending=False))

## 6. Facoltativo: k-means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cols = ["Rotational speed", "Torque", "Tool wear"]
Z = StandardScaler().fit_transform(df[cols])
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
df["gruppo"] = km.labels_
print(df.groupby("gruppo")[cols + ["Machine failure"]].mean())

## Le tue risposte

Rispondi in modo breve ma motivato, citando i numeri che hai ottenuto.

**Domanda 1.** Che accuratezza ottiene il modello pigro? Perché questo numero inganna?

✍️ **La tua risposta:** _scrivi qui_

**Domanda 2.** Dalla matrice di confusione della regressione logistica: quanti guasti trova e quanti ne sfuggono?

✍️ **La tua risposta:** _scrivi qui_

**Domanda 3.** Quale soglia scegli in base ai costi? Motiva con i numeri.

✍️ **La tua risposta:** _scrivi qui_

**Domanda 4.** Scrivi a parole una regola dell’albero che porta a «guasto». Sembra una regola che darebbe un tecnico?

✍️ **La tua risposta:** _scrivi qui_

**Domanda 5.** Chi trova più guasti tra albero e foresta? Quali sensori contano di più?

✍️ **La tua risposta:** _scrivi qui_

## Consegna su Classroom

- Il notebook completato in formato .ipynb, oppure il link in sola lettura
- Le risposte alle 5 domande nelle celle «La tua risposta»

Per scaricare: **File › Scarica › Scarica .ipynb**, poi in Classroom **Aggiungi o crea › File** e **Consegna**.

### Griglia di valutazione

| Criterio | Punti |
| --- | --- |
| Paradosso dell’accuratezza spiegato | 2 |
| Matrice e metriche lette correttamente | 3 |
| Soglia scelta in base ai costi | 2 |
| Albero letto e confronto con la foresta | 3 |
| **Totale** | **10** |

**Suggerimenti:** In scikit-learn la matrice è [[VN, FP], [FN, VP]]: la classe «guasto» è la seconda. Gli alberi non hanno bisogno di normalizzazione: usano X_train non scalato.